In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [2]:
#Configuraciones del LLM --- Ejecutar para definir temperatura y maximo de tokens
generation_config = {
    "temperature": 0.8,
    "top_p": 1,
    "top_k": 1,
    "max_output_tokens": 5000,
}

In [3]:
#importar el modelo GEMINI con el que queremos trabajar
import google.generativeai as genai

# Configurar la API de Gemini (asumiendo que ya se ha configurado anteriormente)
genai.configure(api_key=os.environ['GEMINI_API_KEY'])

# Cargar el modelo Gemini 1.5 Flash
model = genai.GenerativeModel("gemini-1.5-pro-001", generation_config=generation_config)


/Users/gabrielnoguera/Documents/archivo_final/LongContent_Generator_Script/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import json

with open('qa.json', 'r') as file:
    qa = json.load(file)

In [5]:
def cluster_con_gemini(qa):

    # Convertir el JSON a una cadena para incluirlo en el prompt
    qa_str = json.dumps(qa, ensure_ascii=False)

    # Prompt para Gemini
    prompt = f"""
    You are an expert at Semantic SEO. In particular, you are superhuman at taking the result the related searches from user for this topic: {qa_str}
    Your task is to parse these queries and group them into thematic clusters based on their semantic similarities and search context.
    These clusters should represent relevant topics that can be used to organize and better understand user interests.
    Each cluster should include a short descriptive name and a list of queries that belong to that cluster.
    The output format should be a JSON with this structure format:

  'clusters': [
    
      'cluster_name": ‘Cluster name’,
      'queries': [
        'query 1',
        'query 2',
        'query 3'
      ]
    
    Output Lenguage: spanish. 
    Codification: UTF-8 """

    try:
        # Generar la guía con Gemini
        response = model.generate_content(prompt)
        cluster_qa = response.text

        print("Guía para el escritor generada con éxito.")
        return cluster_qa
    except Exception as e:
        print(f"Error al generar la guía con Gemini: {str(e)}")
        return "Error al generar la guía con Gemini"

# Llamar a la función para generar la guía
cluster_de_qa = cluster_con_gemini(qa)

# Imprimir la guía generada
print("\nGuía para el escritor:")
print(cluster_de_qa)


Guía para el escritor generada con éxito.

Guía para el escritor:
```json
{
  'clusters': [
    {
      'cluster_name': 'Autopublicación en Amazon',
      'queries': [
        'amazon kdp',
        'publicar libro amazon papel',
        'como publicar un libro en amazon',
        'es rentable publicar un libro en amazon',
        'amazon kdp españa',
        'como publicar un libro en amazon y ganar dinero',
        'cuanto cuesta publicar un libro en amazon',
        'publicar un libro gratis en amazon',
        'publicar libro en amazon precio',
        'publicar libro en amazon españa',
        'como publicar un libro en amazon gratis',
        'como publicar un libro en amazon paso a paso',
        'amazon kindle',
        'amazon ads',
        'amazon libros',
        'publicar cuento infantil amazon',
        'se puede vivir de vender libros en amazon',
        'amazon kdp precios',
        'amazon kdp mi cuenta',
        'amazon kdp plantillas',
        'amazon kdp opiniones',
 

In [ ]:
def generar_outline_con_gemini(cluster_de_qa):
    prompt = f"""
    Use the following clusterization {cluster_de_qa} and generate an create the structure of a possible article for each one.
    The structure should be organized in HTML hierarchies using H1, H2, H3, etc. headings, and should provide an outline of topics and subtopics to comprehensively cover each cluster.
    Please provide the structure in well organized and hierarchical markdown.
    Output Lenguage: spanish. Codification: UTF-8
    """
    try:
        response = model.generate_content(prompt)
        outline = response.text
        print("Esquema generado con éxito.")
        return outline
    except Exception as e:
        print(f"Error al generar el esquema con Gemini: {str(e)}")
        return "Error al generar el esquema con Gemini"

# Llamar a la función para generar el esquema
esquema_articulo_qa = generar_outline_con_gemini(cluster_de_qa)

# Guardar el esquema en un archivo markdown
with open("esquema_articulo_qa.md", "w", encoding="utf-8") as f:
    f.write(esquema_articulo_qa)

print("\nEsquema del artículo guardado en 'esquema_articulo_qa.md'")




In [ ]:
def generar_contenido_seccion(seccion, esquema_articulo):
    prompt = f"""
    Based on the outline provided, develop the content for the next section of the article.
    The content should develop only what defines the indicated section, should be informative and optimized for SEO.

    Be sure to follow these guidelines:

    1. **Introduction for the main section**: If it is a main heading, start with a brief introduction that summarizes the objective or relevance of the section.
    2. **Subsection development**: For each subsection, provide a detailed and complete development. Prioritize explanatory paragraphs over lists, except if the list is necessary for clarity.
    3. **Expansion of ideas**: Provide specific examples and details to optimize content for SEO, without conclusions or redundancies.

    Article outline:
    {esquema_articulo}

    Section to develop:
    {seccion}

    Output format: Markdown
    Output language: English
    Encoding: UTF-8

    """
    try:
        response = model.generate_content(prompt)
        contenido_seccion = response.text
        print(f"Contenido generado con éxito para la sección: {seccion[:50]}...")
        return contenido_seccion
    except Exception as e:
        print(f"Error al generar contenido para la sección {seccion[:50]}...: {str(e)}")
        return f"Error al generar contenido para la sección: {seccion[:50]}..."
def mejorar_contenido(contenido_seccion):
    prompt = f"""
    Actúa como editor y revisa el siguiente contenido expandiendo las subsecciones.
    
    Asegúrate de:
    1. Ampliar cada subsección con detalles adicionales, ejemplos y contexto donde corresponda.
    2. Convertir listas en párrafos descriptivos, excepto cuando una lista mejore la claridad del contenido.
    3. Revisar y optimizar el contenido para SEO, sin añadir conclusiones.
    4. Varía la estructura y el flujo del texto para:
        a. Diversificar el vocabulario y el estilo
        b. Incorpora expresiones idiomáticas y coloquiales
        c. Introducir errores tipográficos o gramaticales menores
        d. Añadir emociones, humor o sarcasmo (cuando sea apropiado)
        e. Incluir una voz personal
        f. Utilizar transiciones y conectores variados

    Contenido inicial:
    {contenido_seccion}

    Formato de salida: Markdown
    Idioma de salida: Español
    Codificación: UTF-8
    """

    try:
        response = model.generate_content(prompt)
        contenido_mejorado = response.text
        print("Contenido mejorado con éxito.")
        return contenido_mejorado
    except Exception as e:
        print(f"Error al mejorar contenido: {str(e)}")
        return f"Error al mejorar contenido."

def procesar_esquema(esquema_path, guia_para_escritor):
    contenido_mejorado = []
    seccion_actual = []

    with open(esquema_path, 'r', encoding='utf-8') as f:
        for linea in f:
            if linea.startswith("* **"):
                if seccion_actual:
                    seccion_texto = ''.join(seccion_actual).strip()
                    contenido_seccion = generar_contenido_seccion(seccion_texto, guia_para_escritor)
                    contenido_seccion = mejorar_contenido(contenido_seccion)
                    contenido_mejorado.append(contenido_seccion)
                    seccion_actual = []
                seccion_actual.append(linea)
            elif linea.startswith("**"):
                seccion_actual.append(linea)
            elif seccion_actual:
                seccion_actual.append(linea)

    if seccion_actual:
        seccion_texto = ''.join(seccion_actual).strip()
        contenido_seccion = generar_contenido_seccion(seccion_texto, guia_para_escritor)
        contenido_seccion = mejorar_contenido(contenido_seccion)
        contenido_mejorado.append(contenido_seccion)

    return contenido_mejorado

contenido_mejorado = procesar_esquema("esquema_articulo.md", guia_para_escritor)
contenido_completo = "\n\n".join(contenido_mejorado)

print("\nContenido completo generado:")
print(contenido_completo[:500] + "...")

with open("articulo_completo.md", "w", encoding="utf-8") as f:
    f.write(contenido_completo)

print("\nEl artículo completo ha sido guardado en 'articulo_completo.md'")


In [ ]:
import base64
import requests
import os

def publicar_en_wordpress(titulo, archivo_contenido):
    # Obtener credenciales de WordPress desde variables de entorno
    login = os.getenv('WORDPRESS_LOGIN')
    password = os.getenv('WORDPRESS_PASSWORD')

    if not login or not password:
        print("Error: No se encontraron las credenciales de WordPress en las variables de entorno.")
        return

    # Configurar la URL de la API de WordPress y los encabezados de autorización
    url = 'https://archivofinal.com/wp-json/wp/v2/posts'
    headers = {
        'Authorization': 'Basic ' + base64.b64encode(f"{login}:{password}".encode()).decode()
    }

    # Leer el contenido del archivo
    try:
        with open(archivo_contenido, 'r', encoding='utf-8') as file:
            contenido = file.read()
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {archivo_contenido}")
        return
    except IOError:
        print(f"Error: No se pudo leer el archivo {archivo_contenido}")
        return

    # Preparar el cuerpo de la solicitud
    data = {
        'title': titulo,
        'content': contenido,
        'status': 'draft'
    }

    # Realizar la solicitud POST
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        print('Entrada creada correctamente en WordPress como borrador.')
    except requests.exceptions.RequestException as e:
        print(f'Error al crear la entrada en WordPress: {str(e)}')

# Llamar a la función para publicar en WordPress
publicar_en_wordpress("Publicar Novela", "articulo_completo.md")